In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("MP-DE").getOrCreate()

In [0]:
df = spark.read.table('cl_mp_de.`01_bronze`.bronze_return_transaction')
display(df)

### Standardizing column names

In [0]:
df_standarized = df.withColumnRenamed('RTN_ID_No', 'return_id') \
                .withColumnRenamed('Orgnl_Trxn_ID?', 'original_transaction_id') \
                .withColumnRenamed('Rtn_Date_String','return_date') \
                .withColumnRenamed('Rsn_Code_!', 'rsn_code')
display(df_standarized)

### Standardizing return_date

In [0]:
from pyspark.sql.functions import coalesce, expr
#formatting date in 'yyyy-MM-dd'
#casting to date
df_clean = df_standarized.withColumn(
    'return_date',
    coalesce(
        expr("try_to_date(`return_date`, 'dd-MM-yy')"),
        expr("try_to_date(`return_date`, 'yyyy.MM.dd')"),
        expr("try_to_date(`return_date`, 'dd-MMM-yy')"),
        expr("try_to_date(initcap(`return_date`), 'dd-MM-yy')"),
        expr("try_to_date(`return_date`, 'MM/dd/yyyy')"),
    ).cast('date')
)
display(df_clean)

In [0]:
df_clean.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('cl_mp_de.`02_silver`.silver_return_transaction')